# Hyperparameter Tuning

## Objectives
- Load the prepared engineered train, validation, and test splits from `../data/`.
- Tune the baseline tree-based models from notebook 3 with `RandomizedSearchCV`.
- Prefer cuML and GPU-backed estimators where they are available.
- Put extra search budget into the strongest full-feature models, especially XGBoost.
- Evaluate the best searched full-feature models on the held-out test split.


In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the working directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())

## Load Prepared Splits

This section loads both saved training variants from notebook 2: the original scaled split and the SMOTE-balanced split. Validation and test data stay untouched so the later comparisons reflect generalization rather than resampled training behavior.


In [ ]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_train_original = pd.read_csv("../data/X_train_original.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_test = pd.read_csv("../data/X_test.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")
y_train_original = pd.read_csv("../data/y_train_original.csv").squeeze("columns")
y_val = pd.read_csv("../data/y_val.csv").squeeze("columns")
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")

imbalance_ratio = float((y_train_original == 0).sum() / max((y_train_original == 1).sum(), 1))

print("SMOTE X_train shape:", X_train.shape)
print("Original X_train shape:", X_train_original.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Original train neg:pos ratio:", f"{imbalance_ratio:.2f}:1")


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier as SkRandomForestClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

GPU_ENABLED = False

try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
    GPU_ENABLED = True
    print("Using cuML RandomForestClassifier on GPU where available.")
except ImportError:
    cudf = None
    cuRandomForestClassifier = None
    print("cuML not available; using sklearn estimators on CPU.")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGB_AVAILABLE = False
    print(f"XGBoost unavailable in this environment: {exc}")

# cuML takes over the random forest path here so RandomizedSearchCV can try GPU-backed fits when RAPIDS is installed in a Linux/NVIDIA CUDA environment; on macOS this notebook will fall back to CPU instead.
RandomForestClassifier = cuRandomForestClassifier if GPU_ENABLED else SkRandomForestClassifier


## Search Helpers

This section sets up the shared scoring and search utilities. It keeps prediction handling consistent across sklearn, cuML, and XGBoost so the validation and test summaries line up cleanly.


In [ ]:
import numpy as np


def score_predictions(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def normalize_predictions(y_pred):
    if hasattr(y_pred, "to_pandas"):
        y_pred = y_pred.to_pandas()
    elif type(y_pred).__module__.startswith("cupy"):
        y_pred = y_pred.get()
    return np.asarray(y_pred)


def run_random_search(model_name, train_source, estimator, param_distributions, X_train_split, y_train_split, X_val_split, y_val_split, feature_set, n_iter=12):
    X_search = pd.concat([X_train_split, X_val_split], ignore_index=True)
    y_search = pd.concat([y_train_split, y_val_split], ignore_index=True)
    validation_fold = [-1] * len(X_train_split) + [0] * len(X_val_split)
    splitter = PredefinedSplit(test_fold=validation_fold)

    search = RandomizedSearchCV(
        estimator=clone(estimator),
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="f1",
        cv=splitter,
        random_state=12,
        verbose=1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_search, y_search)

    rows = pd.DataFrame(search.cv_results_)
    rows["model"] = model_name
    rows["train_source"] = train_source
    rows["feature_set"] = feature_set
    rows["overfit_gap"] = rows["mean_train_score"] - rows["mean_test_score"]
    rows["params"] = rows["params"].apply(lambda params: dict(params))

    y_val_pred = normalize_predictions(search.best_estimator_.predict(X_val_split))
    best_summary = {
        "model": model_name,
        "train_source": train_source,
        "feature_set": feature_set,
        "params": dict(search.best_params_),
        "overfit_gap": float(rows.loc[rows["rank_test_score"].idxmin(), "overfit_gap"]),
        **score_predictions(y_val_split, y_val_pred),
    }

    return rows, best_summary, search.best_estimator_


## Search Space

This section defines the anti-overfitting search ranges for each model family. The search now compares original-versus-SMOTE training data, favors shallower trees and larger leaves, and gives XGBoost the widest regularized search budget.


In [ ]:
original_scale_weights = sorted({round(max(1.0, imbalance_ratio * factor), 2) for factor in [0.75, 1.0, 1.25, 1.5]})
smote_scale_weights = [1.0, 1.5, 2.0]

search_jobs = [
    {
        "model": "tree",
        "train_source": "Original",
        "estimator": DecisionTreeClassifier(random_state=12),
        "params": {
            "max_depth": [4, 6, 8, 12],
            "min_samples_split": [20, 50, 100],
            "min_samples_leaf": [10, 25, 50],
            "class_weight": ["balanced"],
            "ccp_alpha": [0.0, 0.0005, 0.001, 0.005],
        },
        "n_iter": 16,
    },
    {
        "model": "tree",
        "train_source": "SMOTE",
        "estimator": DecisionTreeClassifier(random_state=12),
        "params": {
            "max_depth": [4, 6, 8, 12],
            "min_samples_split": [20, 50, 100],
            "min_samples_leaf": [10, 25, 50],
            "class_weight": [None],
            "ccp_alpha": [0.0, 0.0005, 0.001, 0.005],
        },
        "n_iter": 16,
    },
    {
        "model": "forest",
        "train_source": "SMOTE",
        "estimator": RandomForestClassifier(n_estimators=400, random_state=12),
        "params": {
            "n_estimators": [200, 400, 600],
            "max_depth": [8, 12, 16, 24],
            "min_samples_split": [20, 50, 100],
            "min_samples_leaf": [5, 10, 20],
            "max_features": [0.3, 0.5, 0.75],
        },
        "n_iter": 20,
    },
    {
        "model": "grad",
        "train_source": "SMOTE",
        "estimator": HistGradientBoostingClassifier(random_state=12),
        "params": {
            "learning_rate": [0.02, 0.05, 0.08],
            "max_depth": [4, 6, 8],
            "max_leaf_nodes": [15, 31, 63],
            "min_samples_leaf": [50, 100, 200],
            "l2_regularization": [0.1, 1.0, 5.0],
            "max_iter": [200, 400, 600],
        },
        "n_iter": 20,
    },
]

if XGB_AVAILABLE:
    xgb_common = {
        "random_state": 12,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
    }
    if GPU_ENABLED:
        xgb_common["device"] = "cuda"

    conservative_xgb_params = {
        "n_estimators": [200, 400, 600, 800],
        "learning_rate": [0.02, 0.05, 0.08],
        "max_depth": [3, 4, 5, 6],
        "min_child_weight": [5, 10, 20],
        "subsample": [0.6, 0.7, 0.8],
        "colsample_bytree": [0.5, 0.7, 0.8],
        "gamma": [1.0, 3.0, 5.0],
        "reg_alpha": [0.1, 1.0, 5.0],
        "reg_lambda": [5.0, 10.0, 20.0],
    }

    search_jobs.extend([
        {
            "model": "xgb",
            "train_source": "Original",
            "estimator": XGBClassifier(**xgb_common),
            "params": {
                **conservative_xgb_params,
                "scale_pos_weight": original_scale_weights,
            },
            "n_iter": 30,
        },
        {
            "model": "xgb",
            "train_source": "SMOTE",
            "estimator": XGBClassifier(**xgb_common),
            "params": {
                **conservative_xgb_params,
                "scale_pos_weight": smote_scale_weights,
            },
            "n_iter": 24,
        },
    ])

if not GPU_ENABLED:
    for spec in search_jobs:
        if spec["model"] == "forest":
            spec["params"]["class_weight"] = [None, "balanced", "balanced_subsample"]


## Randomized Search

This section runs a broader but more conservative search on the full engineered feature set. It favors shallower trees, larger leaves, stronger regularization, and both original-versus-SMOTE training comparisons so the selected models are less likely to overfit the validation split.


In [ ]:
training_sets = {
    "SMOTE": (X_train, y_train),
    "Original": (X_train_original, y_train_original),
}

search_results = []
best_rows = []
best_models = {}

for spec in search_jobs:
    model_name = spec["model"]
    train_source = spec["train_source"]
    X_train_split, y_train_split = training_sets[train_source]

    print(f"Running RandomizedSearchCV for {model_name} on Full features with {train_source} training data...")
    result_df, best_summary, best_model = run_random_search(
        model_name=model_name,
        train_source=train_source,
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        X_train_split=X_train_split,
        y_train_split=y_train_split,
        X_val_split=X_val,
        y_val_split=y_val,
        feature_set="Full",
        n_iter=spec.get("n_iter", 12),
    )
    search_results.append(result_df)
    best_rows.append(best_summary)
    best_models[(train_source, model_name)] = best_model

all_results = pd.concat(search_results, ignore_index=True)
all_results[["model", "train_source", "rank_test_score", "mean_test_score", "overfit_gap", "params"]].sort_values(["model", "train_source", "rank_test_score"]).head(12)


## Best Validation Models

This section collects the strongest searched setup for each model-and-training-source combination. It surfaces the shortlist that moves on to test-set evaluation and later threshold tuning.


In [ ]:
best_validation_models = pd.DataFrame(best_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
best_validation_models


## Test Set Evaluation

This section scores the best searched full-feature setups on the held-out test split using the default prediction threshold. The extra `train_source` and `overfit_gap` columns make it easier to spot which training recipe generalizes more honestly.


In [ ]:
test_rows = []

for _, row in best_validation_models.iterrows():
    model_name = row["model"]
    train_source = row["train_source"]
    params = row["params"]
    model = best_models[(train_source, model_name)]

    y_pred = normalize_predictions(model.predict(X_test))

    test_rows.append({
        "model": model_name,
        "train_source": train_source,
        "feature_set": "Full",
        "params": params,
        "overfit_gap": row["overfit_gap"],
        **score_predictions(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
test_results


## Threshold Tuning

These cells reuse the already-fitted `best_models` from this session. They run a coarse-to-fine threshold sweep on validation predictions, then apply the strongest threshold to the held-out test split without retraining the models.


In [ ]:
def normalize_scores(values):
    if hasattr(values, "to_pandas"):
        values = values.to_pandas()
    elif type(values).__module__.startswith("cupy"):
        values = values.get()
    values = np.asarray(values)
    if values.ndim == 2 and values.shape[1] > 1:
        values = values[:, 1]
    return values.reshape(-1)


def get_model_scores(model, X):
    if hasattr(model, "predict_proba"):
        return normalize_scores(model.predict_proba(X))
    if hasattr(model, "decision_function"):
        return normalize_scores(model.decision_function(X))
    raise AttributeError(f"{type(model).__name__} does not expose predict_proba or decision_function.")


def evaluate_thresholds(y_true, scores, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            **score_predictions(y_true, y_pred),
        })
    return pd.DataFrame(rows)


def find_best_threshold(y_true, scores, thresholds=None):
    coarse_thresholds = thresholds if thresholds is not None else np.linspace(0.05, 0.95, 91)
    coarse_df = evaluate_thresholds(y_true, scores, coarse_thresholds)
    coarse_df = coarse_df.sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)

    coarse_best = float(coarse_df.loc[0, "threshold"])
    fine_min = max(0.01, coarse_best - 0.05)
    fine_max = min(0.99, coarse_best + 0.05)
    fine_thresholds = np.linspace(fine_min, fine_max, 81)

    threshold_df = pd.concat([coarse_df, evaluate_thresholds(y_true, scores, fine_thresholds)], ignore_index=True)
    threshold_df = threshold_df.drop_duplicates(subset=["threshold"]).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
    return threshold_df, threshold_df.iloc[0].to_dict()


In [ ]:
threshold_search_rows = []
threshold_test_rows = []

for _, row in best_validation_models.iterrows():
    model_name = row["model"]
    train_source = row["train_source"]
    params = row["params"]
    model = best_models[(train_source, model_name)]

    val_scores = get_model_scores(model, X_val)
    threshold_df, best_threshold_row = find_best_threshold(y_val, val_scores)
    threshold_df["model"] = model_name
    threshold_df["train_source"] = train_source
    threshold_df["feature_set"] = "Full"
    threshold_df["params"] = str(params)
    threshold_df["overfit_gap"] = row["overfit_gap"]
    threshold_search_rows.append(threshold_df)

    best_threshold = float(best_threshold_row["threshold"])
    test_scores = get_model_scores(model, X_test)
    test_preds = (test_scores >= best_threshold).astype(int)

    threshold_test_rows.append({
        "model": model_name,
        "train_source": train_source,
        "feature_set": "Full",
        "params": params,
        "overfit_gap": row["overfit_gap"],
        "threshold": best_threshold,
        **score_predictions(y_test, test_preds),
    })

threshold_search_results = pd.concat(threshold_search_rows, ignore_index=True)
threshold_test_results = pd.DataFrame(threshold_test_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)

threshold_test_results


## Save Tuning Outputs

This section writes the search summaries, best-model tables, and threshold-tuned results back to `../data/` so the final comparisons are easy to reopen outside the notebook.


In [ ]:
threshold_search_results.to_csv("../data/threshold_search_results.csv", index=False)
threshold_test_results.to_csv("../data/test_results_threshold_tuned_models.csv", index=False)

print("Saved threshold-tuning summaries to ../data/")

In [ ]:
all_results.to_csv("../data/hyperparameter_search_results.csv", index=False)
best_validation_models.to_csv("../data/best_validation_models.csv", index=False)
test_results.to_csv("../data/test_results_tuned_models.csv", index=False)

print("Saved search and test summaries to ../data/")
